# 03 · API Demo (Colab)

**역할:** Service / Demo Developer

Colab → 맥 `localhost:8000` **접근 불가**. ngrok 또는 배포 URL 사용.

In [ ]:
REPO = "https://github.com/toryhyeon80/wind-turbine-yolo.git"
!git clone {REPO} wind-turbine-yolo 2>/dev/null || (cd wind-turbine-yolo && git pull)
%cd wind-turbine-yolo
!pip install -q -r requirements.txt requests

In [ ]:
# 맥에서: uvicorn backend.main:app --host 0.0.0.0 --port 8000
# ngrok http 8000 → Forwarding URL 을 아래에 입력
API_BASE = "https://YOUR-NGROK-OR-DEPLOY-URL"  # 예: https://xxxx.ngrok-free.app
PREDICT_URL = f"{API_BASE.rstrip('/')}/api/v1/predict"

In [ ]:
import requests
from pathlib import Path

sample = next(Path("report/assets/predict").glob("*.jpg"), None)
if sample and "YOUR-NGROK" not in API_BASE:
    with open(sample, "rb") as f:
        r = requests.post(PREDICT_URL, files={"file": (sample.name, f, "image/jpeg")}, timeout=60)
    print(r.status_code)
    print(r.json())
else:
    print("API_BASE 설정 또는 샘플 이미지 없음 — 아래 셀(직접 추론) 사용")

In [ ]:
# API 없이 Colab에서 직접 추론 (대안)
from ultralytics import YOLO
from pathlib import Path

weights = Path("runs/detect/train/weights/best.pt")
if not weights.exists():
    raise FileNotFoundError("best.pt 없음 — 02_train_colab 먼저 실행 또는 Git LFS/pull")
model = YOLO(str(weights))
sample = next(Path("report/assets/predict").glob("*.jpg"))
results = model.predict(str(sample), conf=0.25, device=0)
results[0].show()